<a href="https://colab.research.google.com/github/mediolanum1/dla/blob/2024/CTC_WER_CER_CTC_with_beam_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install pickle
!pip install torch
!pip install editdistance

ERROR: Could not find a version that satisfies the requirement pickle (from versions: none)
ERROR: No matching distribution found for pickle


In [6]:
import torch
import pickle

# CTC decoding

In [7]:
with open("mystery_records.pickle", "rb") as f: # loading pre-computed CTC output
    batch = pickle.load(f)

log_probs = batch["log_probs"] # log probs of softmax layer [batch_size, T, vocab_size]

ind2char = batch["ind2char"] # dict with index to char mapping

EMPTY_TOK = "^"
EMPTY_IND = 0

In [9]:
def ctc_decode(inds, ind2char):
  decoded = []
  last_char_ind = EMPTY_IND
  for ind in inds:
    if last_char_ind == ind:
      continue
    if ind != EMPTY_IND:
      decoded.append(ind2char[ind])
    last_char_ind = ind

  return "".join(decoded)

for i, rec in enumerate(log_probs):
  text = ctc_decode(rec.argmax(-1).numpy(), ind2char) # here argmax takes from log-probs highest value
  print(f"{i}) {text}")

0) we nostrngesto love you know therols and so do i a foll commitment what i thinking of you wolden get this from any ather guy
1)  never gona give you up never donelet you down never go arun around and deset you never gon a make you cri never gonna say good by


# CER, WER metrics

Reminder:

WER - word error rate - number of insertions, deletions and substitutions to get target from prediction. Considered to be main metric

CER - character error rate - same as WER but at char level

Could be the case when WER is high and CER is low and vice versa

In [63]:
import editdistance # library for fast calc of edit distance

def calc_wer(target_text: str, pred_text: str):
  return editdistance.eval(target_text.split(), pred_text.split()) / len(target_text.split())

def calc_cer(target_text: str, pred_text: str):
  return editdistance.eval(target_text, pred_text) / len(target_text)

In [28]:
import numpy as np

for target, pred, expected_wer, expected_cer in [
  ("if you can not measure it you can not improve it",
     "if you can nt measure t yo can not i",
     0.454, 0.25),
    ("if you cant describe what you are doing as a process you dont know what youre doing",
     "if you cant describe what you are doing as a process you dont know what youre doing",
     0.0, 0.0),
    ("one measurement is worth a thousand expert opinions",
     "one  is worth thousand opinions",
     0.375, 0.392)
]:
    wer = calc_wer(target, pred)
    cer = calc_cer(target, pred)
    print(cer, " ", expected_cer, " ------ ", wer , " ", expected_wer)
    assert np.isclose(wer, expected_wer, atol=1e-3), f"true: {target}, pred: {pred}, expected wer {expected_wer} != your wer {wer}"
    assert np.isclose(cer, expected_cer, atol=1e-3), f"true: {target}, pred: {pred}, expected cer {expected_cer} != your cer {cer}"


0.25   0.25  ------  0.45454545454545453   0.454
0.0   0.0  ------  0.0   0.0
0.39215686274509803   0.392  ------  0.375   0.375


Task: come up with pred and target so WER > 1 and CER > WER

In [73]:
target, pred = "Well, everybodys talkin bout the stormy weather And whats a man to do", "We will, every bundle stalking boat short and winter andy watt amen so we aredo"
assert calc_wer(target, pred) > 1

target, pred =  "a a a ", "To the extentt that I weer skirtts and cheaap nylon slipps I've gone native"
print(calc_cer(target, pred), calc_wer(target, pred))
assert calc_cer(target, pred) > calc_wer(target, pred)

11.5 4.666666666666667


# Beam Search

In [90]:
with open('lj_batch.pickle', 'rb') as f:
  batch = pickle.load(f)

log_probs = batch['log_probs']

ind2char = batch['ind2char']

true_texts = batch['text']

In [79]:
probs = log_probs.exp()
log_probs

tensor([[[-2.1086e-04, -1.3408e+01, -1.5260e+01,  ..., -1.4973e+01,
          -1.8417e+01, -1.9708e+01],
         [-1.5378e-05, -1.6262e+01, -1.8436e+01,  ..., -1.6892e+01,
          -2.2445e+01, -2.3296e+01],
         [-2.1458e-06, -1.7606e+01, -1.9730e+01,  ..., -1.8801e+01,
          -2.5158e+01, -2.5966e+01],
         ...,
         [-1.0729e-06, -1.6843e+01, -1.8346e+01,  ..., -1.9335e+01,
          -2.5702e+01, -1.6679e+01],
         [-6.8343e-04, -1.3386e+01, -1.4756e+01,  ..., -1.5950e+01,
          -2.2557e+01, -7.3026e+00],
         [-3.7174e-01, -7.6812e+00, -8.5722e+00,  ..., -8.6173e+00,
          -1.5256e+01, -1.1772e+00]],

        [[-7.1642e-05, -1.3834e+01, -1.4518e+01,  ..., -1.5474e+01,
          -2.0535e+01, -2.4722e+01],
         [-6.9141e-06, -1.6142e+01, -1.5708e+01,  ..., -1.6346e+01,
          -2.3528e+01, -2.8610e+01],
         [-7.1526e-07, -1.8094e+01, -1.7141e+01,  ..., -1.8898e+01,
          -2.7023e+01, -3.1732e+01],
         ...,
         [-1.1921e-06, -1

In [87]:
from collections import defaultdict # default dict is part of default python library and helps with providing default value
                                    # for missing values in dict
from tqdm import tqdm

# This function expands and merges paths based on the next character probabilities.
# params: dp - dict storing possible paths(prefixes) and their probs up to current timestep
#         next_token_probs - probs of next possible chars

# basically creates dict and stores all possible paths, for each next_token_prob compares with our last char to avoid reps
# in the end multiplies current prob v with next_token_prob that we choose
def expand_and_merge_path(dp, next_token_probs, ind2char):
  new_dp = defaultdict(float)
  for ind, next_token_prob in enumerate(next_token_probs):
    current_char = ind2char[ind]
    for (prefix, last_char), v in dp.items():
      if last_char == current_char:
        new_prefix = prefix
      else:
        if current_char != EMPTY_TOK:
          new_prefix = prefix + current_char
        else:
          new_prefix = prefix
      new_dp[(new_prefix, current_char)] += v * next_token_prob
  return new_dp

# This function keeps only the top beam_size paths with the highest probabilities.
def truncate_paths(dp, beam_size):
  return dict(sorted(list(dp.items()), key=lambda x: -x[1])[:beam_size])


def ctc_beam_search(probs, beam_size, ind2char):
  dp = {
      ('', EMPTY_TOK): 1.0,  # dp is initialized with a single path ('', EMPTY_TOK) and probability 1.0.
  }
  for prob in probs:
    dp = expand_and_merge_path(dp, prob, ind2char)
    dp = truncate_paths(dp,beam_size)
  dp = [(prefix, proba) for (prefix, _), proba in sorted(dp.items(), key = lambda x: -x[1])]
  return dp

bs_results = []
for log_probs_line in log_probs:
  bs_results.append(ctc_beam_search(log_probs_line.exp().numpy(), 100, ind2char))

In [88]:
bs_results[0][:]

[('he wl ge to her iand tell her all hisan ly omblications',
  1.2623606088745703e-10),
 ('he wl ge to her and tell her all hisan ly omblications',
  1.2342274269227618e-10),
 ('he wl ge to her iand tell her all hisanly omblications',
  1.128661362665549e-10),
 ('he wl ge to her and tell her all hisanly omblications',
  1.1035078247187708e-10),
 ('he wl ge to her iand tell her all hisan ly omblocations',
  1.0519466150320952e-10),
 ('he wl ge to her iand tell her all hisanely omblications',
  1.0426635970834307e-10),
 ('he wl ge to her and tell her all hisan ly omblocations',
  1.0285027549209407e-10),
 ('he wl ge to her and tell her all hisanely omblications',
  1.0194266198797231e-10),
 ('he wl ge to her iand tell her all hisanly omblocations',
  9.405327539743516e-11),
 ('he wld ge to her iand tell her all hisan ly omblications',
  9.221707229145529e-11),
 ('he wl ge to her and tell her all hisanly omblocations',
  9.195718820070406e-11),
 ('he wl ge to her iand tell her all hisanel

In [91]:
for i in range(len(true_texts)):
    beam_search_hypos = bs_results[i][:3]
    true_text = true_texts[i]
    argmax_text = ctc_decode(log_probs[i].numpy().argmax(-1), ind2char)
    print("True: ", true_text)
    print(f"Argmax: {argmax_text} --- (CER: {calc_cer(true_text, argmax_text):.3f})")
    for ind, (hypo, score) in enumerate(beam_search_hypos):
        print(f"{ind+1}) '{hypo}' --- (CER: {calc_cer(true_text, hypo):.3f})")
    print('-' * 100)


True:  he would go to her and tell her all his family complications
Argmax: he wld ge toher iand tell her all mhisan ly omblications --- (CER: 0.200)
1) 'he wl ge to her iand tell her all hisan ly omblications' --- (CER: 0.183)
2) 'he wl ge to her and tell her all hisan ly omblications' --- (CER: 0.167)
3) 'he wl ge to her iand tell her all hisanly omblications' --- (CER: 0.183)
----------------------------------------------------------------------------------------------------
True:  he did not say the last as a boast but merely as an assurance to the liveryman who he saw was anxious on his account
Argmax: he did not sad the last is a bost but mearlioves an asurance to the livery man who re saw was anxes on his account --- (CER: 0.129)
1) 'he did not say the last is a bost but merli oves an a surance to the livery man who re saw was anxes on his account' --- (CER: 0.112)
2) 'he did not say the last as a bost but merli oves an a surance to the livery man who re saw was anxes on his acc